In [1]:
import numpy as np

# Gets board at next step if agent drops piece in selected column
def drop_piece(grid, col, piece, config):
    next_grid = grid.copy()
    for row in range(config.rows-1, -1, -1):
        if next_grid[row][col] == 0:
            break
    next_grid[row][col] = piece
    return next_grid

# Returns True if dropping piece in column results in game win
def check_winning_move(obs, config, col, piece):
    # Convert the board to a 2D grid
    grid = np.asarray(obs.board).reshape(config.rows, config.columns)
    next_grid = drop_piece(grid, col, piece, config)
    # horizontal
    for row in range(config.rows):
        for col in range(config.columns-(config.inarow-1)):
            window = list(next_grid[row,col:col+config.inarow])
            if window.count(piece) == config.inarow:
                return True
    # vertical
    for row in range(config.rows-(config.inarow-1)):
        for col in range(config.columns):
            window = list(next_grid[row:row+config.inarow,col])
            if window.count(piece) == config.inarow:
                return True
    # positive diagonal
    for row in range(config.rows-(config.inarow-1)):
        for col in range(config.columns-(config.inarow-1)):
            window = list(next_grid[range(row, row+config.inarow), range(col, col+config.inarow)])
            if window.count(piece) == config.inarow:
                return True
    # negative diagonal
    for row in range(config.inarow-1, config.rows):
        for col in range(config.columns-(config.inarow-1)):
            window = list(next_grid[range(row, row-config.inarow, -1), range(col, col+config.inarow)])
            if window.count(piece) == config.inarow:
                return True
    return False

In [2]:
from kaggle_environments import evaluate, make

CONFIG = {'rows': 6, 'columns': 7, 'inarow': 4}

def get_win_percentages(agent1, agent2, n_rounds=100):
    # Agent 1 goes first (roughly) half the time          
    outcomes = evaluate("connectx", [agent1, agent2], CONFIG, [], n_rounds//2)
    # Agent 2 goes first (roughly) half the time      
    outcomes += [[b,a] for [a,b] in evaluate("connectx", [agent2, agent1], CONFIG, [], n_rounds-n_rounds//2)]
    
    print(f"Agent 1 Win Percentage: {np.round(outcomes.count([1,-1])/len(outcomes), 2)}")
    print(f"Agent 2 Win Percentage: {np.round(outcomes.count([-1,1])/len(outcomes), 2)}")
    print(f"Number of Invalid Plays by Agent 1: {outcomes.count([None, 0])}")
    print(f"Number of Invalid Plays by Agent 2: {outcomes.count([0, None])}")

2025-01-24 23:35:01.047859: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737761701.081661   66529 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737761701.091788   66529 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-24 23:35:01.122238: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import random

def agent_random(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    return random.choice(valid_moves)

In [4]:
# Create an agent that:
# - Selects the winning move, if it is available. (If there is more than one move that lets the agent win the game, the agent can select any of them.)
# - Otherwise, it should select a random move.

def agent_q1(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    
    winning_moves = [move for move in valid_moves if check_winning_move(obs, config, move, obs.mark)]
    if winning_moves:
        return random.choice(winning_moves)
    
    return random.choice(valid_moves)



In [5]:
# Amend the code to create an agent that can also block its opponent from winning. In particular, your agent should:
# - Select a winning move, if one is available.
# - Otherwise, it selects a move to block the opponent from winning, if the opponent has a move that it can play in its next turn to win the game.
# - If neither the agent nor the opponent can win in their next moves, the agent selects a random move.

def agent_q2(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    
    winning_moves = [move for move in valid_moves if check_winning_move(obs, config, move, obs.mark)]
    if winning_moves:
        return random.choice(winning_moves)
        
    blocking_moves = [move for move in valid_moves if check_winning_move(obs, config, move, 1 + obs.mark % 2)]
    if blocking_moves:
        return random.choice(blocking_moves)
        
    return random.choice(valid_moves)


In [6]:
def my_agent(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    
    winning_moves = [move for move in valid_moves if check_winning_move(obs, config, move, obs.mark)]
    if winning_moves:
        return random.choice(winning_moves)
        
    blocking_moves = [move for move in valid_moves if check_winning_move(obs, config, move, 1 + obs.mark % 2)]
    if blocking_moves:
        return random.choice(blocking_moves)
        
    return random.choice(valid_moves)

In [7]:
#get_win_percentages(agent1=agent_q2, agent2=agent_random)

In [8]:
# env = make("connectx", debug=True)
# env.run([my_agent, agent_q2])
# env.render(mode="ipython")

In [9]:
#get_win_percentages(agent1=agent_random, agent2=agent_q2)

In [10]:
get_win_percentages(agent1=agent_q2, agent2=my_agent)

Agent 1 Win Percentage: 0.51
Agent 2 Win Percentage: 0.44
Number of Invalid Plays by Agent 1: 0
Number of Invalid Plays by Agent 2: 0


In [11]:
get_win_percentages(agent1=my_agent, agent2=agent_q2)

Agent 1 Win Percentage: 0.45
Agent 2 Win Percentage: 0.48
Number of Invalid Plays by Agent 1: 0
Number of Invalid Plays by Agent 2: 0


In [12]:
# Agents play one game round
env = make("connectx", debug=True)
env.run([my_agent, agent_q2])
env.render(mode="ipython")

In [13]:
import inspect
import os

def write_agent_to_file(function, file):
    with open(file, "a" if os.path.exists(file) else "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(my_agent, "submission.py")

<function my_agent at 0x7f1e138b5e40> written to submission.py
def my_agent(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]

    winning_moves = [move for move in valid_moves if check_winning_move(obs, config, move, obs.mark)]
    blocking_moves = [move for move in valid_moves if check_winning_move(obs, config, move, 1 + obs.mark % 2)]
    
    if blocking_moves:
        return random.choice(blocking_moves)
    if winning_moves:
        return random.choice(winning_moves)        
        
    return random.choice(valid_moves)

